In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

In [ ]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   # --- CONFIG_MODEL_MODEL_MODEL_MODELURATION -                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   --
# C'est ici qu'on règle la sensibilité du mod                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   èle
CONFIG_MODEL = {               
    'WINDOW_RECENT_DAYS': 14,    # Période "c                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   ourt terme" (tendance récente)
    'WINDOW_HISTORY_DAYS': 14,  # Période "mo                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   yen terme" (pour comparaison)
    'TARGET_WINDOW': 10,        # Fenêtre de  prédiction (churn dans les X prochains jours)
    'STEP_SIZE': 7,             # On avance de 7 jours à chaque itération (Data Augmentation)
    'CHURN_EVENT': 'Cancellation Confirmation'
}

In [3]:
def parse_os(user_agent):
    """Extrait le système d'exploitation du User Agent"""
    if pd.isna(user_agent): return 'Unknown'
    ua = str(user_agent).lower()
    if 'iphone' in ua or 'ipad' in ua: return 'iOS'
    if 'macintosh' in ua or 'mac os' in ua: return 'Mac'
    if 'windows' in ua: return 'Windows'
    if 'linux' in ua or 'android' in ua: return 'Linux/Android'
    return 'Other'



In [4]:
def load_and_clean_data(csv_path):
    print("Chargement et Feature Engineering 'Statique'...")
    df = pd.read_parquet(csv_path)

    # 1. Dates
    df['ts_date'] = pd.to_datetime(df['ts'], unit='ms', errors='coerce')
    # La registration est souvent aussi un timestamp ms
    if 'registration' in df.columns:
        df['reg_date'] = pd.to_datetime(df['registration'], unit='ms', errors='coerce')

    # 2. Tri
    df = df.sort_values(['userId', 'ts_date']).reset_index(drop=True)
    df = df[df['userId'].notna()]

    # 3. Extraction OS (Device) - On le fait une fois ici
    if 'userAgent' in df.columns:
        df['os_type'] = df['userAgent'].apply(parse_os).astype('category')

    # 4. Conversion types
    categorical_cols = ['gender', 'level', 'page', 'method']
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    print(f"Données prêtes : {df.shape[0]} lignes.")
    return df

In [5]:
df=pd.read_parquet("../../../data/churn-prediction-25-26/train.parquet")

In [6]:
def compute_features_at_anchor(df_window, anchor_date):
    """
    V5.1 CORRIGÉE : Gestion robuste des événements 'Error' et 'Roll Advert' depuis la colonne 'page'.
    """
    WINDOW_RECENT = 7
    WINDOW_HISTORY = 28

    start_recent = anchor_date - timedelta(days=WINDOW_RECENT)
    start_history = anchor_date - timedelta(days=WINDOW_HISTORY)

    # Filtrage global
    df_past = df_window[(df_window['ts_date'] >= start_history) & (df_window['ts_date'] <= anchor_date)].copy()
    if df_past.empty: return None

    # --- 1. GAP ANALYSIS ---
    df_past = df_past.sort_values(['userId', 'ts_date'])
    df_sessions = df_past.drop_duplicates(subset=['userId', 'sessionId'], keep='last').copy()
    df_sessions['prev_ts'] = df_sessions.groupby('userId')['ts_date'].shift(1)
    df_sessions['gap_days'] = (df_sessions['ts_date'] - df_sessions['prev_ts']).dt.total_seconds() / (3600 * 24)

    user_gap_stats = df_sessions.groupby('userId')['gap_days'].agg(['mean', 'last']).rename(columns={
        'mean': 'avg_gap_days',
        'last': 'last_gap_days'
    })

    # --- 2. ANALYSE DE LA DERNIÈRE SESSION (CORRECTION ICI) ---
    # On isole la dernière session de chaque user
    last_sessions = df_past.sort_values('ts_date').groupby('userId').last()['sessionId']
    df_last_sess = df_past[df_past['sessionId'].isin(last_sessions)]

    # A. Durée et Chansons (Ça marche car 'length' et 'song' existent)
    base_last_stats = df_last_sess.groupby('userId').agg({
        'length': 'sum',
        'song': 'count'
    }).rename(columns={'length': 'last_sess_duration', 'song': 'last_sess_songs'})

    # B. Events spécifiques (La correction : on utilise crosstab sur 'page' au lieu de chercher des colonnes)
    # On compte combien de fois chaque page apparait dans la dernière session
    last_sess_events = pd.crosstab(df_last_sess['userId'], df_last_sess['page'])

    # On extrait les colonnes qui nous intéressent (si elles existent, sinon 0)
    if 'Error' in last_sess_events.columns:
        base_last_stats['last_sess_errors'] = last_sess_events['Error']
    else:
        base_last_stats['last_sess_errors'] = 0

    if 'Roll Advert' in last_sess_events.columns:
        base_last_stats['last_sess_ads'] = last_sess_events['Roll Advert']
    else:
        base_last_stats['last_sess_ads'] = 0

    # --- 3. DERNIÈRE ACTION ABSOLUE ---
    last_actions = df_past.sort_values('ts_date').groupby('userId').last()['page']
    last_exit_features = pd.DataFrame(index=df_past['userId'].unique())
    last_exit_features['exit_on_ad'] = (last_actions == 'Roll Advert').astype(int)
    last_exit_features['exit_on_error'] = (last_actions == 'Error').astype(int)
    last_exit_features['exit_on_thumbs_down'] = (last_actions == 'Thumbs Down').astype(int)

    # --- 4. RATIO D'ENNUI ---
    artist_stats = df_past.groupby('userId').agg({'artist': 'nunique', 'song': 'count'})
    epsilon = 1
    artist_stats['exploration_ratio'] = artist_stats['artist'] / (artist_stats['song'] + epsilon)

    # --- 5. CALCUL DU ZAPPING ---
    song_stats = df_past[df_past['page'] == 'NextSong'].groupby('userId')['length'].agg(['mean', 'std']).rename(columns={
        'mean': 'avg_song_duration',
        'std': 'std_song_duration'
    })

    # --- 6. ASSEMBLAGE FINAL ---
    mask_recent = df_past['ts_date'] >= start_recent
    df_recent = df_past[mask_recent]
    df_history = df_past

    unique_users = df_past['userId'].unique()
    features = pd.DataFrame(index=unique_users)

    # Jointure des blocs calculés plus haut
    features = features.join(user_gap_stats).fillna(0)
    features['gap_acceleration'] = (features['last_gap_days'] + 0.1) / (features['avg_gap_days'] + 0.1)

    features = features.join(base_last_stats, how='left')
    features = features.join(last_exit_features, how='left')
    features = features.join(artist_stats[['exploration_ratio']], how='left')
    features = features.join(song_stats, how='left')

    # --- 7. HELPER ROBUSTE POUR RECENT/HISTORY ---
    def get_event_stats(df_sub, prefix):
        for col in ['page', 'song', 'sessionId', 'length']:
            if col not in df_sub.columns: df_sub[col] = 0

        stats = df_sub.groupby('userId').agg({
            'page': 'count', 'song': 'count', 'length': 'sum', 'sessionId': 'nunique'
        })

        target_events = ['Roll Advert', 'Thumbs Down', 'Thumbs Up', 'Error', 'Add Friend', 'Submit Downgrade', 'Upgrade']
        events = df_sub[df_sub['page'].isin(target_events)]

        if not events.empty:
            pv = pd.crosstab(events['userId'], events['page'])
            pv = pv.reindex(columns=target_events, fill_value=0)
            stats = stats.join(pv, how='left').fillna(0)
        else:
            for col in target_events: stats

In [7]:
def compute_features_at_anchor(df_window, anchor_date):
    """
    V4 ULTIME : Intègre le 'Zapping' (Skip rate proxy), la densité de session et les Downgrades.
    """
    WINDOW_RECENT = 7  # On repasse à 7 jours pour lisser un peu le bruit
    WINDOW_HISTORY = 28

    start_recent = anchor_date - timedelta(days=WINDOW_RECENT)
    start_history = anchor_date - timedelta(days=WINDOW_HISTORY)

    # Filtrage global
    df_past = df_window[(df_window['ts_date'] >= start_history) & (df_window['ts_date'] <= anchor_date)].copy()
    if df_past.empty: return None

    # --- 1. GAP ANALYSIS (Conservé car crucial) ---
    df_past = df_past.sort_values(['userId', 'ts_date'])
    df_sessions = df_past.drop_duplicates(subset=['userId', 'sessionId'], keep='last').copy()
    df_sessions['prev_ts'] = df_sessions.groupby('userId')['ts_date'].shift(1)
    df_sessions['gap_days'] = (df_sessions['ts_date'] - df_sessions['prev_ts']).dt.total_seconds() / (3600 * 24)

    user_gap_stats = df_sessions.groupby('userId')['gap_days'].agg(['mean', 'last']).rename(columns={
        'mean': 'avg_gap_days',
        'last': 'last_gap_days'
    })

    # --- 2. CALCUL DU ZAPPING (NOUVEAU) ---
    # On considère qu'une chanson très courte (< 60s) ou une moyenne faible indique du zapping
    # (Note: Ce dataset a déjà des lengths calculées, on regarde la moyenne par user)
    song_stats = df_past[df_past['page'] == 'NextSong'].groupby('userId')['length'].agg(['mean', 'std']).rename(columns={
        'mean': 'avg_song_duration',
        'std': 'std_song_duration'
    })

    # --- 3. SEPARATION RECENT / HISTORY ---
    mask_recent = df_past['ts_date'] >= start_recent
    df_recent = df_past[mask_recent]
    df_history = df_past

    unique_users = df_past['userId'].unique()
    features = pd.DataFrame(index=unique_users)

    # Joindre les stats calculées
    features = features.join(user_gap_stats).join(song_stats).fillna(0)

    # Ratio d'accélération du silence
    features['gap_acceleration'] = (features['last_gap_days'] + 0.1) / (features['avg_gap_days'] + 0.1)

    # --- 4. EVENTS SPECIFIQUES ---
    def get_event_stats(df_sub, prefix):
        # Initialisation
        for col in ['page', 'song', 'sessionId', 'length']:
            if col not in df_sub.columns: df_sub[col] = 0

        stats = df_sub.groupby('userId').agg({
            'page': 'count',
            'song': 'count', # Nb chansons
            'length': 'sum', # Temps total
            'sessionId': 'nunique'
        })

        # Events critiques
        target_events = ['Roll Advert', 'Thumbs Down', 'Thumbs Up', 'Error', 'Add Friend', 'Submit Downgrade', 'Upgrade']

        events = df_sub[df_sub['page'].isin(target_events)]
        if not events.empty:
            pv = pd.crosstab(events['userId'], events['page'])
            pv = pv.reindex(columns=target_events, fill_value=0)
            stats = stats.join(pv, how='left').fillna(0)
        else:
            for col in target_events: stats[col] = 0

        stats = stats.add_prefix(f'{prefix}_')
        return stats

    feats_recent = get_event_stats(df_recent, 'recent')
    feats_history = get_event_stats(df_history, 'history')

    features = features.join(feats_recent, how='left').join(feats_history, how='left').fillna(0)

    # --- 5. RATIOS PSYCHOLOGIQUES (NOUVEAU) ---
    epsilon = 0.01

    # A. Zapping Récent vs Historique
    # On calcule la durée moyenne d'une écoute récente vs historique
    avg_len_recent = features['recent_length'] / (features['recent_song'] + epsilon)
    avg_len_history = features['history_length'] / (features['history_song'] + epsilon)

    # Si Ratio < 1 : Il écoute des morceaux plus courts qu'avant (il zappe ?)
    features['trend_song_duration'] = avg_len_recent / (avg_len_history + epsilon)

    # B. Intensité de Session (Combien de temps il reste par session ?)
    avg_sess_len_recent = features['recent_length'] / (features['recent_sessionId'] + epsilon)
    avg_sess_len_history = features['history_length'] / (features['history_sessionId'] + epsilon)

    # Si Ratio < 1 : Il fait des sessions de plus en plus courtes (Désengagement)
    features['trend_session_intensity'] = avg_sess_len_recent / (avg_sess_len_history + epsilon)

    # C. Frustration & Interactions
    features['ads_per_hour'] = features['history_Roll Advert'] / ((features['history_length'] / 3600) + epsilon)
    features['errors_per_session'] = features['history_Error'] / (features['history_sessionId'] + epsilon)

    # D. Le signal "Downgrade"
    # A-t-il essayé de downgrader récemment ? (Signal très fort de churn futur)
    features['recent_downgrade_attempt'] = (features['recent_Submit Downgrade'] > 0).astype(int)

    # Profil
    last_info = df_past.groupby('userId').last()
    if 'level' in last_info.columns: features['is_paid'] = (last_info['level'] == 'paid').astype(int)

    if 'reg_date' in last_info.columns:
        features['tenure_days'] = (anchor_date - last_info['reg_date']).dt.days
    else:
        features['tenure_days'] = 0

    last_action_date = df_past.groupby('userId')['ts_date'].max()
    features['days_since_last_action'] = (anchor_date - last_action_date).dt.total_seconds() / (3600 * 24)

    features['anchor_date'] = anchor_date
    return features

In [8]:
def run_feature_engineering_pipeline(df_logs):
    """
    Exécute la boucle Sliding Window et génère le dataset Train/Test complet.
    """
    print("Démarrage de la pipeline Sliding Window...")

    min_date = df_logs['time'].min()
    max_date = df_logs['time'].max()

    # On commence quand on a assez d'historique
    start_anchor = min_date + timedelta(days=CONFIG_MODEL['WINDOW_HISTORY_DAYS'] + CONFIG_MODEL['WINDOW_RECENT_DAYS'])
    # On s'arrête avant la fin pour avoir la vérité terrain (Target)
    end_anchor = max_date - timedelta(days=CONFIG_MODEL['TARGET_WINDOW'])

    current_date = start_anchor
    final_datasets = []

    while current_date <= end_anchor:
        print(f"Traitement de la fenêtre : {current_date.date()}")

        # 1. Générer les features (X) pour cette date
        # On passe une vue large des données pour éviter les copies inutiles
        window_start = current_date - timedelta(days=CONFIG_MODEL['WINDOW_HISTORY_DAYS'] + CONFIG_MODEL['WINDOW_RECENT_DAYS'] + 1)
        df_view = df_logs[(df_logs['time'] >= window_start) & (df_logs['time'] <= max_date)]

        features = compute_features_at_anchor(df_view, current_date)

        if features is None or features.empty:
            current_date += timedelta(days=CONFIG_MODEL['STEP_SIZE'])
            continue

        # 2. Générer la Target (Y) - Futur
        # On regarde dans le futur [Anchor, Anchor + 10 jours]
        future_mask = (df_view['time'] > current_date) & \
                      (df_view['time'] <= current_date + timedelta(days=CONFIG_MODEL['TARGET_WINDOW']))

        future_data = df_view[future_mask]

        # Identification des churners
        churn_users = future_data[future_data['page'] == CONFIG_MODEL['CHURN_EVENT']]['userId'].unique()

        features['target_churn'] = 0
        features.loc[features.index.isin(churn_users), 'target_churn'] = 1

        final_datasets.append(features)

        # Avancer
        current_date += timedelta(days=CONFIG_MODEL['STEP_SIZE'])

    # Concaténation finale
    if not final_datasets:
        print("Attention: Aucune fensêtre valide générée (dataset trop court ?)")
        return pd.DataFrame()

    full_dataset = pd.concat(final_datasets).reset_index().rename(columns={'index': 'userId'})

    print(f"Pipeline terminée. Dataset généré : {full_dataset.shape}")
    return full_dataset

In [9]:
df_train_ready = run_feature_engineering_pipeline(df)

Démarrage de la pipeline Sliding Window...
Traitement de la fenêtre : 2018-10-29


KeyError: 'ts_date'

In [ ]:


print(df_train_ready['target_churn'].value_counts())

df_train_ready.head()

In [ ]:
df_train_ready[df_train_ready["userId"]=="1697168"]

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

def objective(trial):
    # 1. Définir l'espace de recherche (Hyperparameters)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10), # Très important pour le churn

        # Params fixes
        'objective': 'binary:logistic',
        'eval_metric': 'auc_pr',
        'n_jobs': -1,
        'random_state': 42,
        'early_stopping_rounds': 50
    }

    # 2. Cross Validation (Rapide: 3 folds pour l'optimisation)
    gkf = GroupKFold(n_splits=3)
    scores = []

    # Préparation des données (Assure-toi que df_train_ready est chargé)
    X = df_train_ready.drop(columns=['userId', 'target_churn', 'anchor_date', 'ts_date'], errors='ignore')
    y = df_train_ready['target_churn']
    groups = df_train_ready['userId']

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        clf = xgb.XGBClassifier(**params)

        # Note: early_stopping_rounds est passé dans le constructeur (XGBoost moderne)
        # Mais pour la compatibilité avec .fit() dans la boucle optuna, on peut le laisser gérer par l'objet
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = clf.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))

    return np.mean(scores)

# --- LANCEMENT DE L'OPTIMISATION ---
print("Démarrage de l'optimisation Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # 30 essais (augmente à 50 ou 100 si tu as le temps)

print("\n MEILLEURS PARAMETRES TROUVÉS :")
print(study.best_params)

# --- UTILISATION DES MEILLEURS PARAMETRES ---
best_params = study.best_params
# Tu utiliseras ces params dans ta fonction train_eval_model finale !

In [ ]:
def train_eval_model(df_train, best_params=None):
    """
    Entraîne le modèle avec les meilleurs hyperparamètres trouvés par Optuna.
    :param best_params: Dictionnaire des paramètres optimaux (ou None pour utiliser les défauts)
    """
    print("--- Démarrage de l'entraînement du modèle Optimisé ---")

    # 1. Préparation
    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    features = [c for c in df_train.columns if c not in ignore_cols]

    X = df_train[features]
    y = df_train['target_churn']
    groups = df_train['userId']

    print(f"Features utilisées ({len(features)})")

    # 2. Configuration des Paramètres
    # Paramètres par défaut (Filet de sécurité si tu ne passes rien)
    final_params = {
        'n_estimators': 500,
        'learning_rate': 0.02,
        'max_depth': 4,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': 10, # Sera écrasé si Optuna en a trouvé un meilleur
        'eval_metric': 'auc',
        'n_jobs': -1,
        'random_state': 42,
        'early_stopping_rounds': 50
    }

    # --- LA MAGIE EST ICI ---
    # Si on fournit des best_params, on met à jour la configuration
    if best_params:
        print(f"Application des hyperparamètres Optuna : {best_params}")
        final_params.update(best_params)

    # 3. Initialisation du modèle avec "Unpacking" (**final_params)
    clf = xgb.XGBClassifier(**final_params)

    # 4. Cross-Validation
    gkf = GroupKFold(n_splits=5)
    fold_scores = []

    for i, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = clf.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, preds)
        fold_scores.append(score)
        print(f"Fold {i+1} AUC: {score:.4f}")

    print(f"\n>>> SCORE FINAL MOYEN (AUC): {np.mean(fold_scores):.4f} <<<")

    # 5. Entraînement Final sur TOUT le dataset
    print("Ré-entraînement final sur l'ensemble des données...")

    # IMPORTANT : On désactive l'early_stopping car pas de validation set ici
    clf.set_params(early_stopping_rounds=None)

    clf.fit(X, y, verbose=False)

    return clf, features

In [ ]:
model, features=train_eval_model(df_train_ready)

In [ ]:
def generate_kaggle_submission(model, df_logs_full, feature_cols, all_test_user_ids, output_file='submission.csv', seuil=0.5):
    """
    Génère la soumission en format BINAIRE avec correction automatique des types d'IDs.
    """
    print("\n--- Génération du fichier de soumission ROBUSTE (Mode Binaire) ---")

    # 1. Date de référence
    last_date = df_logs_full['time'].max()
    print(f"Anchor Date : {last_date}")

    # 2. Calcul des features
    X_test_active = compute_features_at_anchor(df_logs_full, last_date)

    if X_test_active is None:
        X_test_active = pd.DataFrame()
        print("Attention : Aucun utilisateur actif sur la dernière fenêtre !")

    # 3. Prédiction
    if not X_test_active.empty:
        for col in feature_cols:
            if col not in X_test_active.columns:
                X_test_active[col] = 0
        X_test_active = X_test_active[feature_cols]

        active_probs = model.predict_proba(X_test_active)[:, 1]

        df_preds = pd.DataFrame({
            'userId': X_test_active.index,
            'prediction': active_probs
        })
    else:
        df_preds = pd.DataFrame(columns=['userId', 'prediction'])

    # 4. Fusion avec la liste complète
    final_submission = pd.DataFrame({'userId': all_test_user_ids})

    # --- CORRECTION ICI : HARMONISATION DES TYPES ---
    # On convertit les deux colonnes en String pour être sûr qu'elles matchent
    final_submission['userId'] = final_submission['userId'].astype(str)
    df_preds['userId'] = df_preds['userId'].astype(str)

    # On retire les doublons éventuels pour éviter d'exploser le nombre de lignes
    final_submission = final_submission.drop_duplicates(subset=['userId'])
    df_preds = df_preds.drop_duplicates(subset=['userId'])

    # Maintenant le merge va fonctionner
    final_submission = final_submission.merge(df_preds, on='userId', how='left')

    # 5. Gestion des manquants
    avg_proba = final_submission['prediction'].mean()
    if pd.isna(avg_proba): avg_proba = 0.5

    missing_count = final_submission['prediction'].isna().sum()
    print(f"Utilisateurs inactifs : {missing_count} (Remplis avec la proba moyenne : {avg_proba:.4f})")

    final_submission['prediction'] = final_submission['prediction'].fillna(avg_proba)

    # 6. Application du Seuil
    final_submission['is_churn'] = (final_submission['prediction'] >= seuil).astype(int)

    # 7. Sauvegarde
    output_df = final_submission[['userId', 'is_churn']]
    output_df.columns = ['id', 'target']

    # On s'assure que le fichier final n'a pas d'index inutile
    output_df.to_csv(output_file, index=False)
    print(f"Sauvegardé : {output_file} ({len(output_df)} lignes)")

    return output_df

In [ ]:
sample = pd.read_csv('example_submission.csv')
all_users_list = sample['id'].unique()
all_users_list

In [ ]:
test_set=load_and_clean_data("test.parquet")

In [ ]:
submit_df = generate_kaggle_submission(model, test_set, features, all_users_list, seuil=0.26)